In [0]:
# configure paths
catalog = "global_mart_retail_dev"
bronze_table = "superstore_orders"
silver_table = "customer"

# table path
bronze_path = f"{catalog}.bronze.{bronze_table}"
silver_path = f"{catalog}.silver.{silver_table}"

print(f"Bronze table path: {bronze_path}")
print(f"Silver table path: {silver_path}")


In [0]:
# import libraries
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
from pyspark.sql.functions import regexp_replace
# read bronze data 
bronze_df = spark.read.table(bronze_path)

# Check which columns have the replacement character
bronze_df.filter(
    col("Customer_Name").contains("�") 
).display()


#  select and clean (deuplicate) data for customers table 
df_clean_customers =(
    bronze_df
    .select(upper(trim(col("Customer_ID"))).alias("customer_id"),
            upper(trim(col("Customer_Name"))).alias("customer_name"), # to avoid edge cases
            coalesce(lower(trim(col("Segment"))),lit("unknown")).alias("customer_segment"),
            ).dropDuplicates(["customer_id"]
    )
)


print(f"total number of customers: {df_clean_customers.count()}")
display(df_clean_customers.limit(5))


In [0]:
#  Log data quality metrics
print(f"Total customers processed: {bronze_df.select('Customer_ID').distinct().count()}")
print(f"After cleaning: {df_clean_customers.count()}")
print(f"Records with NULL customer_id (excluded): {bronze_df.filter(col('Customer_ID').isNull()).count()}")

In [0]:
# generate hash on cleaned values to track scd type 2

df_customer_stage = (
    df_clean_customers
    .withColumn("customer_hash", sha2(concat_ws("|",col("customer_id"),col("customer_name"),col("customer_segment")),256))
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to", lit("9999-12-31").cast("timestamp"))
    .withColumn("is_current_flag",lit(True))
    .withColumn("load_timestamp", current_timestamp())

)

display(df_customer_stage)

In [0]:
# Create the silver customer table if it does not exist
spark.sql(
    """
        CREATE TABLE IF NOT EXISTS global_mart_retail_dev.silver.customer
        (
            customer_id string not null,
            customer_name string,
            customer_segment string,
            customer_hash string not null,
            valid_from timestamp not null,
            valid_to timestamp,
            is_current_flag boolean not null,
            load_timestamp timestamp not null
        )
        USING DELTA
    """
)


# Set up SCD Type 2 merge for customer table
customer_table = DeltaTable.forName(spark,"global_mart_retail_dev.silver.customer")

(
    customer_table.alias("target")
    .merge(
        df_customer_stage.alias("source"),
        "target.customer_id = source.customer_id AND target.is_current_flag = true"
    )
    # Expire current records when any attributes have changed
    .whenMatchedUpdate(
        condition="target.customer_hash <> source.customer_hash",
        set={
            "valid_to": "current_timestamp()",
            "is_current_flag": "false"
        }
    )
    # Insert new records or updated records
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "customer_segment": "source.customer_segment",
            "customer_hash": "source.customer_hash",
            "valid_from": "source.valid_from",
            "valid_to": "source.valid_to",
            "is_current_flag": "source.is_current_flag",
            "load_timestamp": "source.load_timestamp"
        }
    )
    .execute()
)

In [0]:
%sql
-- sanity check 
select * from global_mart_retail_dev.silver.customer
where customer_id = 'AH-10690' or customer_id = 'DP-13240'


In [0]:
%sql
select count(*) from global_mart_retail_dev.silver.customer
